# MACE+Graph2Mat

This notebook will show you how to integrate a `MACE` model with `Graph2Mat` through the python API. Note that you can also use `MACE+Graph2Mat` through the Command Line Interface (CLI).

Prerequisites
-------------
Before reading this notebook, **make sure you have read the [notebook on computing a matrix](<./Computing a matrix.ipynb>) and [the notebook on batching](./Batching.ipynb)**, which introduce the basic concepts of `graph2mat` that we are going to assume are already known. Also **we will use exactly the same setup as in the batching notebook**, with the only difference that we will add target matrices to each structure.

In [1]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [ ]:
import numpy as np
import pandas as pd
import torch

# To load plotly templates for sisl visualization
import sisl.viz

from e3nn import o3

from graph2mat import (
    BasisConfiguration,
    PointBasis,
    BasisTableWithEdges,
    MatrixDataProcessor,
)
from graph2mat.bindings.torch import TorchBasisMatrixDataset, TorchBasisMatrixData

from graph2mat.bindings.e3nn import E3nnGraph2Mat

from graph2mat.tools.viz import plot_basis_matrix


from torch_geometric.loader import DataLoader

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


Generating a dataset
--------------------

We generate a dataset here just as we have done in the other notebooks.

In [ ]:
# The basis
point_1 = PointBasis("A", R=2, basis="0e", basis_convention="spherical", matrix_role='row')  # "0e"
point_2 = PointBasis("A", R=2, basis="2x0e", basis_convention="spherical", matrix_role='col')
point_3 = PointBasis("B", R=5, basis="0e + 1o", basis_convention="spherical", matrix_role='row')
point_4 = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical", matrix_role='col')


# The basis table.
table = BasisTableWithEdges([point_1, point_2, point_3, point_4])

# The data processor.
processor = MatrixDataProcessor(
    basis_table=table, symmetric_matrix=False,  # Matrix is not square
    sub_point_matrix=False
)

positions = np.array([[0, 0, 0], [6.0, 0, 0], [9, 0, 0]])

config1 = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions,
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

config2 = BasisConfiguration(
    point_types=["B", "A", "B"],
    positions=positions,
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

configs = [config1, config2]

dataset = TorchBasisMatrixDataset(configs, data_processor=processor)


loader = DataLoader(dataset, batch_size=2)

data = next(iter(loader))

Defined row for type A with basis ((1, 0, 1),) and reach 2.
Defined col for type A with basis ((2, 0, 1),) and reach 2.
Defined row for type B with basis ((1, 0, 1), (1, 1, -1)) and reach 5.
Defined col for type B with basis ((2, 0, 1), (1, 1, -1)) and reach 5.
BasisTableWithEdges: is_square = False
BasisTableWithEdges: all matrix roles = ['row', 'col', 'row', 'col']
Basis sizes: [1 4]
Basis sizes: [2 5]
Row cutoff radii: [2 5]
Col cutoff radii: [2 5]
Max cutoff radius: [2 5]
In BasisTableWithEdges: 
self.edge_type == point_types_to_edge_types:
[[ 0  1]
 [-1  2]]
self.point_block_shape:
[[1 4]
 [2 5]]
self.point_block_size:
[ 2 20]
Row basis sizes: [1 4]
Col basis sizes: [2 5]
Point type to edge type:
[[ 0  1]
 [-1  2]]
Edge type to point types:
[[0 0]
 [0 1]
 [1 1]]
Edge block shape:
[[1 1 4]
 [2 5 5]]
Edge block shape inv:
[[1 4 4]
 [2 2 5]]
self.basis_table.R is an array: [2 5]
point_types: [0 1 0]
self.basis_table.R[point_types]: [2 5 2]
Cutoff: [1.9999 4.9999 1.9999]
In BasisMatri

Initializing a MACE model
-------------------------

We will now initialize a normal MACE model.

Note that you must have MACE installed, which you can do with:

```
pip install mace_torch
```

In [4]:
from mace.modules import MACE, RealAgnosticResidualInteractionBlock

num_interactions = 3
hidden_irreps = o3.Irreps("1x0e + 1x1o")

mace_model = MACE(
    r_max=10,
    num_bessel=10,
    num_polynomial_cutoff=10,
    max_ell=2,  # 1,
    interaction_cls=RealAgnosticResidualInteractionBlock,
    interaction_cls_first=RealAgnosticResidualInteractionBlock,
    num_interactions=num_interactions,
    num_elements=2,
    hidden_irreps=hidden_irreps,
    MLP_irreps=o3.Irreps("2x0e"),
    atomic_energies=torch.tensor([0, 0]),
    avg_num_neighbors=2,
    atomic_numbers=[0, 1],
    correlation=2,
    gate=None,
)

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/mace/modules/blocks.py:312: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

Now, we can pass our data through the mace model. MACE outputs many things, but we are just interested in the node features, which we can get from the `"node_feats"` key.

In [5]:
mace_output = mace_model(data)
mace_output["node_feats"]

tensor([[ 0.1095,  0.0000,  0.0000,  0.0046,  0.0402,  0.0000,  0.0000,  0.0008,
         -0.0438],
        [-0.2999,  0.0000,  0.0000,  0.0294, -0.0452,  0.0000,  0.0000,  0.0319,
         -0.0234],
        [ 0.1105,  0.0000,  0.0000, -0.0126,  0.0407,  0.0000,  0.0000, -0.0027,
         -0.0423],
        [-0.2999,  0.0000,  0.0000,  0.0229, -0.0459,  0.0000,  0.0000,  0.0250,
         -0.0269],
        [ 0.1104,  0.0000,  0.0000,  0.0077,  0.0407,  0.0000,  0.0000,  0.0015,
         -0.0410],
        [-0.2999,  0.0000,  0.0000, -0.0523, -0.0457,  0.0000,  0.0000, -0.0566,
         -0.0249]], grad_fn=<CatBackward0>)

Our `Graph2Mat` model will take these node features and convert them to a matrix. Therefore we need to know what its irreps are, and then initialize the `Graph2Mat` module.

In [6]:
# MACE outputs as node features the hidden irreps for each interaction, except
# in the last interaction, where it computes just scalar features.
mace_out_irreps = hidden_irreps * (num_interactions - 1) + str(hidden_irreps[0])

# Initialize the matrix model with this information
matrix_model = E3nnGraph2Mat(
    unique_basis=table,
    irreps=dict(node_feats_irreps=mace_out_irreps),
    symmetric=False,  # Matrix is not square
    # We would need to also implement passing the edge information in order to use
    # preprocessing_edges. As shown later, graph2mat can do this automatically for you.
    preprocessing_edges=None,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/

Now, we can use the matrix model, passing the node features computed by MACE:

In [7]:
node_labels, edge_labels = matrix_model(data=data, node_feats=mace_output["node_feats"])

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 4]
 [2 5]]
shapes_inv:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25  2  3
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45  4  5 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False,  True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True, False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([False,  True

And plot the obtained matrices:

In [8]:
matrices = processor.matrix_from_data(
    data,
    predictions={"node_labels": node_labels, "edge_labels": edge_labels},
)

for config, matrix in zip(configs, matrices):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In processing.py matrix_from_data:
data: TorchBasisMatrixDataBatch(
  metadata={ data_processor=[2] },
  edge_index=[2, 10],
  num_nodes=6,
  neigh_isc=[10],
  n_edges=[2],
  positions=[6, 3],
  shifts=[10, 3],
  cell=[6, 3],
  nsc=[2, 3],
  node_attrs=[6, 2],
  point_types=[6],
  edge_types=[10],
  batch=[6],
  ptr=[3]
)
is_batch: True
In MatrixDataProcessor.yield_from_batch:
arrays=data.numpy_arrays(): <graph2mat.core.data.processing.NumpyArraysProvider object at 0x76e9d47e09d0>
atom_ptr: [0 3 6]
edge_ptr: [ 0  4 10]
In BasisTableWithEdges.point_block_pointer:
  point_types = [0 1 0 1 0 1]
  point_block_size = [ 2 20]
  pointers = [ 0  2 22 24 44 46 66]
In BasisTableWithEdges.edge_block_pointer:
  edge_types = [ 1 -1  1 -1  1 -1  1 -1  2 -2]
  edge_block_size = [ 2  5 20]
  pointers = [ 0  5 13 18 26 31 39 44 52 72 92]
example 0 (batch):
  atom_start: 0
  atom_end: 3
  edge_start: 0
  edge_end: 4
  new_edge_label = edge_labels[edge_labels_ptr[edge_start]: edge_labels_ptr[edge_end]]:


Using MatrixMACE
----------------

If you don't want to handle the details of interacting `MACE` with `Graph2Mat`, you can also use `MatrixMACE`, which takes a mace model and wraps it to also output the `node_labels` and `edge_labels` corresponding to a matrix. 

Internally, it just initializes a `E3nnGraph2Mat` layer. However it can handle the interaction between `MACE` and `Graph2Mat` in more complex cases like having an extra preprocessing step for edges, which needs some extra inputs from MACE.

In [9]:
from graph2mat.models import MatrixMACE
from graph2mat.bindings.e3nn import E3nnEdgeMessageBlock

In [10]:
matrix_mace_model = MatrixMACE(
    mace_model,
    unique_basis=table,
    readout_per_interaction=True,
    edge_hidden_irreps=o3.Irreps("10x0e + 10x1o + 10x2e"),
    symmetric=False,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.1

The output of this model is MACE's output plus the `node_labels` and `edge_labels` for the predicted matrix:

In [11]:
out = matrix_mace_model(data)

out

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 4]
 [2 5]]
shapes_inv:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25  2  3
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45  4  5 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False,  True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True, False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([False,  True

{'energy': tensor([-0.0718, -0.7290], grad_fn=<SumBackward1>),
 'node_energy': tensor([ 0.1951, -0.4630,  0.1962, -0.4618,  0.1955, -0.4626],
        grad_fn=<SumBackward1>),
 'contributions': tensor([[ 0.0000,  0.0000, -0.1230,  0.0083,  0.0430],
         [ 0.0000,  0.0000, -0.7535, -0.0118,  0.0364]],
        grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],
 
         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[ 0.1095,  0.0000,  0.0000,  0.0046,  0.0402,  0.0000,  0.0000,  0.0008,
          -0.0438],
         [-0.2999,  0.0000,  0.0000,  0.0294, -0.0452,  0.0000,  0.0000,  0.0319,
          -0.0234],
         [ 0.1105,  0.0000,  0.0000, -0.0126,  0.0407,  0.0000,  0.0000, -0.0027,
          -0.0423],
         [-0.2999,  0.0000,  0.

You can of course plot the predicted matrices:

In [12]:
matrices = processor.matrix_from_data(data, predictions=out)

for config, matrix in zip(configs, matrices):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In processing.py matrix_from_data:
data: TorchBasisMatrixDataBatch(
  metadata={ data_processor=[2] },
  edge_index=[2, 10],
  num_nodes=6,
  neigh_isc=[10],
  n_edges=[2],
  positions=[6, 3],
  shifts=[10, 3],
  cell=[6, 3],
  nsc=[2, 3],
  node_attrs=[6, 2],
  point_types=[6],
  edge_types=[10],
  batch=[6],
  ptr=[3]
)
is_batch: True
In MatrixDataProcessor.yield_from_batch:
arrays=data.numpy_arrays(): <graph2mat.core.data.processing.NumpyArraysProvider object at 0x76e9d704e500>
atom_ptr: [0 3 6]
edge_ptr: [ 0  4 10]
In BasisTableWithEdges.point_block_pointer:
  point_types = [0 1 0 1 0 1]
  point_block_size = [ 2 20]
  pointers = [ 0  2 22 24 44 46 66]
In BasisTableWithEdges.edge_block_pointer:
  edge_types = [ 1 -1  1 -1  1 -1  1 -1  2 -2]
  edge_block_size = [ 2  5 20]
  pointers = [ 0  5 13 18 26 31 39 44 52 72 92]
example 0 (batch):
  atom_start: 0
  atom_end: 3
  edge_start: 0
  edge_end: 4
  new_edge_label = edge_labels[edge_labels_ptr[edge_start]: edge_labels_ptr[edge_end]]:


# Rotating matrix

A matrix should rotate equivariantly if we rotate the configuration given to mace via the Data. Let us try!

Thing sthat remain constant under rotation: the basis we defined and the thing sthat depend on it.

- table object: an processed object with the basis of each point basis (and the basis points point_1, point_2, point_3, point_4)
- data_processor object processor : has info of how to process the basis, i.e., information about the edges and pointers.
- mace_model and matrix_mace_model : it is just the architecture of the mace model, so we use the same model for both the original and rotated configurations. Matrixmace is just the matrixed version of the mace model, so it is also the same for both configurations.

In [13]:
positions_rot = np.array([[0, 0, 0], [0.0, 6.0, 0], [0, 9.0, 0]])

config1_rot = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions_rot,  # changed positions to rotated ones
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

config2_rot = BasisConfiguration(
    point_types=["B", "A", "B"],
    positions=positions_rot,  # changed positions to rotated ones
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

configs_rot = [config1_rot, config2_rot]

dataset_rot = TorchBasisMatrixDataset(configs_rot, data_processor=processor)


loader_rot = DataLoader(dataset_rot, batch_size=2)

data_rot = next(iter(loader_rot))

self.basis_table.R is an array: [2 5]
point_types: [0 1 0]
self.basis_table.R[point_types]: [2 5 2]
Cutoff: [1.9999 4.9999 1.9999]
In BasisMatrixData.from_config 1:
edge_index: [[0 1 1 2]
 [1 0 2 1]]
edge_types: [ 1 -1 -1  1]
In sort_edge_index:
isc_off: [[[0]]]
sc_shifts: [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
isc: [0 0 0 0]
In BasisMatrixData.from_config 2:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]
In BasisMatrixData.from_config 3:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]
self.basis_table.R is an array: [2 5]
point_types: [1 0 1]
self.basis_table.R[point_types]: [5 2 5]
Cutoff: [4.9999 1.9999 4.9999]
In BasisMatrixData.from_config 1:
edge_index: [[0 0 1 1 2 2]
 [1 2 0 2 0 1]]
edge_types: [-1  2  1  1  2 -1]
In sort_edge_index:
isc_off: [[[0]]]
sc_shifts: [[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]
isc: [0 0 0 0 0 0]
In BasisMatrixData.from_config 2:
edge_index: [[1 0 1 2 0 2]
 [0 1 2 1 2 0]]
edge_types: [ 1 -1  1 -1  2 -2]
In BasisMatrixData.from_c

In [14]:
out_rot = matrix_mace_model(data_rot)

out_rot

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 4]
 [2 5]]
shapes_inv:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25  2  3
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45  4  5 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1,  1, -1,  1, -1,  2, -2], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False,  True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True, False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([False,  True

{'energy': tensor([-0.0718, -0.7290], grad_fn=<SumBackward1>),
 'node_energy': tensor([ 0.1951, -0.4630,  0.1962, -0.4618,  0.1955, -0.4626],
        grad_fn=<SumBackward1>),
 'contributions': tensor([[ 0.0000,  0.0000, -0.1230,  0.0083,  0.0430],
         [ 0.0000,  0.0000, -0.7535, -0.0118,  0.0364]],
        grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],
 
         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[ 0.1095,  0.0046,  0.0000,  0.0000,  0.0402,  0.0008,  0.0000,  0.0000,
          -0.0438],
         [-0.2999,  0.0294,  0.0000,  0.0000, -0.0452,  0.0319,  0.0000,  0.0000,
          -0.0234],
         [ 0.1105, -0.0126,  0.0000,  0.0000,  0.0407, -0.0027,  0.0000,  0.0000,
          -0.0423],
         [-0.2999,  0.0229,  0.

You can of course plot the predicted matrices:

In [15]:
matrices_rot = processor.matrix_from_data(data_rot, predictions=out_rot)

for config, matrix in zip(configs_rot, matrices_rot):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In processing.py matrix_from_data:
data: TorchBasisMatrixDataBatch(
  metadata={ data_processor=[2] },
  edge_index=[2, 10],
  num_nodes=6,
  neigh_isc=[10],
  n_edges=[2],
  positions=[6, 3],
  shifts=[10, 3],
  cell=[6, 3],
  nsc=[2, 3],
  node_attrs=[6, 2],
  point_types=[6],
  edge_types=[10],
  batch=[6],
  ptr=[3]
)
is_batch: True
In MatrixDataProcessor.yield_from_batch:
arrays=data.numpy_arrays(): <graph2mat.core.data.processing.NumpyArraysProvider object at 0x76e9d704ce80>
atom_ptr: [0 3 6]
edge_ptr: [ 0  4 10]
In BasisTableWithEdges.point_block_pointer:
  point_types = [0 1 0 1 0 1]
  point_block_size = [ 2 20]
  pointers = [ 0  2 22 24 44 46 66]
In BasisTableWithEdges.edge_block_pointer:
  edge_types = [ 1 -1  1 -1  1 -1  1 -1  2 -2]
  edge_block_size = [ 2  5 20]
  pointers = [ 0  5 13 18 26 31 39 44 52 72 92]
example 0 (batch):
  atom_start: 0
  atom_end: 3
  edge_start: 0
  edge_end: 4
  new_edge_label = edge_labels[edge_labels_ptr[edge_start]: edge_labels_ptr[edge_end]]:


Summary and next steps
----------------------

In this notebook we learned **how to interface MACE with Graph2Mat**.

The **next steps** could be:

- **Train a MACE+Graph2Mat model** following the steps in [this notebook](<./Fitting matrices.ipynb>), replacing the model by the `MACE+Graph2Mat` model.